In [1]:
%pip install -q langchain langchain-community langchain-huggingface langchain-core sentence_transformers langchain-chroma pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /mimer/NOBACKUP/groups/naiss2025-23-515/yingyi/venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Part 1: The dataset
## Task 1.1. Downloading and inspecting the question answering dataset

In [7]:
!wget https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json

--2026-06-03 22:12:09--  https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2584787 (2.5M) [text/plain]
Saving to: ‘ori_pqal.json.2’

ori_pqal.json.2     100%[===================>]   2.46M  --.-KB/s    in 0.04s   

2026-06-03 22:12:10 (60.8 MB/s) - ‘ori_pqal.json.2’ saved [2584787/2584787]



In [8]:
import pandas as pd
tmp_data = pd.read_json("ori_pqal.json").T
# some labels have been defined as "maybe", only keep the yes/no answers
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])]

documents = pd.DataFrame({"abstract": tmp_data.apply(lambda row: (" ").join(row.CONTEXTS+[row.LONG_ANSWER]), axis=1),
             "year": tmp_data.YEAR})
questions = pd.DataFrame({"question": tmp_data.QUESTION,
             "year": tmp_data.YEAR,
             "gold_label": tmp_data.final_decision,
             "gold_context": tmp_data.LONG_ANSWER,
             "gold_document_id": documents.index})

### Sanity Check

In [11]:
print("documents shape:", documents.shape)
print("questions shape:", questions.shape)

print("\nExample question:")
print(questions.iloc[0].question)

print("\nGold label:")
print(questions.iloc[0].gold_label)

print("\nGold document id:")
print(questions.iloc[0].gold_document_id)

print("\nExample document:")
print(documents.iloc[0].abstract[:1000])

print("\nSanity check 1.1: PASSED")

documents shape: (890, 2)
questions shape: (890, 5)

Example question:
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Gold label:
yes

Gold document id:
21645374

Example document:
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on 

# Step 2: Configure your LangChain LM
## Task 2.1. Select a language model

In [12]:
%pip install -q transformers accelerate


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /mimer/NOBACKUP/groups/naiss2025-23-515/yingyi/venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
from langchain_huggingface import HuggingFacePipeline

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

model = HuggingFacePipeline.from_model_id(
    model_id=model_id,
    task="text-generation",
    model_kwargs={
        "device_map": "auto",
        "torch_dtype": "auto",
    },
    pipeline_kwargs={
        "max_new_tokens": 100,
        "do_sample": False,
        "return_full_text": False,
    },
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

### Sanity Check

In [20]:
prompt = "How is Sweden? Answer:"

response = model.invoke(prompt)
print(response)

print("\nSanity check 2.1: PASSED")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Sweden is a country in Northern Europe. It has a population of about 10 million people and its capital city is Stockholm. The official language is Swedish, which is also the most widely spoken Scandinavian language. Sweden is known for its high standard of living, strong welfare state, and emphasis on education and culture. It is also famous for its natural beauty, including forests, lakes, and mountains. In terms of economy, Sweden is one of the wealthiest countries in the world, with a highly developed

Sanity check 2.1: PASSED


# Part 3: Set up the document database
## Task 3.1. Embedding mode

In [23]:
from langchain_huggingface import HuggingFaceEmbeddings
import torch
import numpy as np

embedding_model_id = "BAAI/bge-small-en-v1.5"

embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_id,
    model_kwargs={
        "device": "cuda" if torch.cuda.is_available() else "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Sanity Check

In [25]:
test_text = "How is Sweden? Answer:"

embedding = embedding_model.embed_query(test_text)

print(type(embedding))
print(len(embedding))
print(np.array(embedding).shape)
print(embedding[:5])

<class 'list'>
384
(384,)
[0.03405091166496277, -0.0452423132956028, -0.01178905088454485, 0.008442137390375137, 0.07452898472547531]


## Task 3.2. Chunking

In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

metadatas = [{"id": idx} for idx in documents.index]
texts = text_splitter.create_documents(texts=documents.abstract.tolist(), metadatas=metadatas)

chunks = text_splitter.split_documents(texts)

### Sanity Check

In [32]:
print("Number of original documents:", len(documents))
print("Number of LangChain documents before splitting:", len(texts))
print("Number of chunks after splitting:", len(chunks))

for i in range(3):
    print("=" * 80)
    print("Chunk index:", i)
    print("Metadata:", chunks[i].metadata)
    print("Length:", len(chunks[i].page_content))
    print(chunks[i].page_content[:1000])

from collections import Counter

chunk_counts = Counter([chunk.metadata["id"] for chunk in chunks])

print("Min chunks per document:", min(chunk_counts.values()))
print("Max chunks per document:", max(chunk_counts.values()))
print("Average chunks per document:", sum(chunk_counts.values()) / len(chunk_counts))

print("\nFirst 10 document chunk counts:")
for doc_id, count in list(chunk_counts.items())[:10]:
    print(doc_id, count)

print("\nSanity check 3.2: PASSED")

Number of original documents: 890
Number of LangChain documents before splitting: 1946
Number of chunks after splitting: 1946
Chunk index: 0
Metadata: {'id': 21645374}
Length: 913
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not un

## Task 3.3. Define a vector store

In [35]:
from langchain_chroma import Chroma

persist_dir = "./pubmedqa_chroma_db"

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="pubmedqa_rag",
    persist_directory=persist_dir,
    collection_metadata={"hnsw:space": "cosine"}
)

### Sanity Check

In [39]:
results = vector_store.similarity_search_with_score(
    "What is programmed cell death?", k=3
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

print("\nSanity check 3.3: PASSED")

* [SIM=0.209706] Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD) [{'id': 21645374}]
* [SIM=0.333140] . No activation-related mutations

# Part 4: Implementing the system##  Task 4.1. Defining the full RAG pipelin (Option B)e

In [41]:
from operator import itemgetter
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

retriever = vector_store.as_retriever(
    search_kwargs={"k": 1}
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_template = """
You are answering a medical question using the provided context.

Use the context to answer the question.
Answer with only one word: Yes or No.

Context:
{context}

Question:
{question}

Answer:
"""

rag_prompt = PromptTemplate.from_template(rag_template)

rag_inputs = RunnableParallel(
    context=RunnableLambda(itemgetter("question")) | retriever,
    question=RunnableLambda(itemgetter("question"))
)

answer_chain = (
    {
        "context": RunnableLambda(itemgetter("context")) | RunnableLambda(format_docs),
        "question": RunnableLambda(itemgetter("question"))
    }
    | rag_prompt
    | model
    | StrOutputParser()
)

rag_chain = rag_inputs.assign(answer=answer_chain)

### Sanity Check

In [43]:
sample_question = questions.iloc[0].question

rag_output = rag_chain.invoke({
    "question": sample_question
})

print("Question:")
print(rag_output["question"])

print("\nRetrieved document metadata:")
print(rag_output["context"][0].metadata)

print("\nRetrieved context:")
print(rag_output["context"][0].page_content[:1500])

print("\nRAG answer:")
print(rag_output["answer"])

print("\nSanity check 4.1: PASSED")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Retrieved document metadata:
{'id': 21645374}

Retrieved context:
. Window stage leaves were stained with the mitochondrial dye MitoTracker Red CMXRos and examined. Mitochondrial dynamics were delineated into four categories (M1-M4) based on characteristics including distribution, motility, and membrane potential (ΔΨm). A TUNEL assay showed fragmented nDNA in a gradient over these mitochondrial stages. Chloroplasts and transvacuolar strands were also examined using live cell imaging. The possible importance of mitochondrial permeability transition pore (PTP) formation during PCD was indirectly examined via in vivo cyclosporine A (CsA) treatment. This treatment resulted in lace plant leaves with a significantly lower number of perforations compared to controls, and that displayed mitochondrial dynamics similar to that of non-PCD cells. Results depicted mitochondrial dynamics in vivo as 

# Part 5: Evaluate RAG on the dataset

## Task 5.1. High-level evaluation

In [46]:
def ask_rag(question):
    output = rag_chain.invoke({"question": question})
    return {
        "question": output["question"],
        "answer": output["answer"].strip(),
        "context": output["context"],
        "retrieved_document_id": output["context"][0].metadata["id"] if len(output["context"]) > 0 else None
    }

### Sanity Check